# MotionHiFlow — Official T2M Benchmark Pipeline

This notebook is the MotionHiFlow model-side implementation for the shared
capability-oriented T2M benchmark. It keeps the upstream MotionHiFlow repository
unchanged and uses two team-controlled inputs:

- `benchmark_definition_v1.0.json` — the shared benchmark definition and source of truth.
- `motionhiflow_utils.py` — MotionHiFlow file lookup, standardisation, rendering and
  optional Matching Score utilities.

The default run evaluates the **18-prompt Pilot split**. The prompts are generated in
three separate batches because the official target lengths are 100, 140 and 180 frames.
Every result retains its official ID, such as `C1-01`, rather than receiving a local
`P001` identifier.

## Output contract

The hand-off package contains:

- 18 standardised `[T, 22, 3]` joint arrays;
- 20 fps, metres, `+Y` up, `+Z` forward and `+X` right;
- initial root `XZ = (0, 0)` and ground plane `Y = 0`;
- a standardisation report, run manifest and framework hand-off manifest;
- optional GIFs and length-stratified Matching Scores.

## Recommended workflow

1. Start a fresh Colab GPU runtime.
2. Run Steps 1-3, then restart the session when instructed.
3. Continue from Step 4 and prepare all resources.
4. Upload the two shared benchmark files in Step 7.
5. Run the optional quick test, then the official Pilot batch.
6. Validate, render, optionally score, and download the hand-off package.

Only the notebook, utility module and small summary files belong in GitHub. Do not
commit checkpoints, HumanML3D data, generated `.npy` files, GIF collections or videos.


## 1. Check runtime

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    # Without a GPU the generation step is extremely slow or unusable.
    print("WARNING: No GPU detected. Switch the Colab runtime to GPU.")

# This workflow was verified on Python 3.11.
if sys.version_info[:2] != (3, 11):
    print("\nWARNING: Python version differs from the verified 3.11 environment; "
          "the dependency pins below may need adjusting.")

## 2. Clone MotionHiFlow

Idempotent: reuse an existing clone, or remove a non-repository directory and clone again.

In [ ]:
%cd /content

from pathlib import Path

REPO = Path("/content/MotionHiFlow")

# run.sh marks a valid repository root.
if not (REPO / "run.sh").exists():
    if REPO.exists():
        print("Directory exists but is not the repository. Removing it first.")
        !rm -rf /content/MotionHiFlow
    !git clone https://github.com/ai-lh/MotionHiFlow.git

%cd /content/MotionHiFlow

# Integrity check: failing fast here is better than a confusing error several steps later.
REQUIRED_REPO_FILES = ["run.sh", "gen_t2m.py", "eval.py", "prepare.sh", "src", "configs"]
missing_repo_files = [x for x in REQUIRED_REPO_FILES if not Path(x).exists()]

if missing_repo_files:
    raise RuntimeError(f"Repository is incomplete. Missing: {missing_repo_files}")

print("MotionHiFlow repository is ready.")

## 3. Install compatible dependencies

**Run this once.** When it finishes choose **Runtime → Restart session**, then continue
from Step 4. `huggingface-hub==0.34.4` is pinned here only once — do not upgrade it later.

In [ ]:
%cd /content/MotionHiFlow

# Versions verified in this Colab reproduction.
%pip install -q --no-cache-dir "numpy==1.26.4" "scipy==1.11.1"

# Keep Colab's built-in torch / torchvision — reinstalling them easily breaks the CUDA
# match — and install only the remaining project requirements.
!grep -vE '^(--extra-index-url|torch==|torchvision==|numpy==|scipy==|transformers($|==)|peft($|==)|tokenizers($|==)|accelerate($|==)|huggingface-hub==|diffusers==)' \
    requirements.txt > requirements_colab.txt

%pip install -q -r requirements_colab.txt

# The HuggingFace stack versions must match each other, otherwise HybridCache and
# tokenizer import errors appear after the restart.
%pip install -q --no-cache-dir \
    "transformers==4.53.2" \
    "peft==0.17.1" \
    "tokenizers==0.21.4" \
    "accelerate==1.10.1" \
    "huggingface-hub==0.34.4" \
    "diffusers==0.35.1"

print("Installation finished.")
print(">>> Now restart the Colab session, then continue from Step 4.")

## 4. Verify imports after restart

Run this **after restarting the runtime**.

In [ ]:
%cd /content/MotionHiFlow

import numpy as np
import scipy
import torch
import transformers
import peft
import diffusers
import huggingface_hub

print("NumPy            :", np.__version__)
print("SciPy            :", scipy.__version__)
print("PyTorch          :", torch.__version__)
print("Transformers     :", transformers.__version__)
print("PEFT             :", peft.__version__)
print("Diffusers        :", diffusers.__version__)
print("Hugging Face Hub :", huggingface_hub.__version__)
print("CUDA available   :", torch.cuda.is_available())

# HybridCache exists at this path only in transformers 4.53.x — a quick version probe.
from transformers import HybridCache  # noqa: F401

assert huggingface_hub.__version__ == "0.34.4", "Unexpected huggingface-hub version."

print("\nImports OK.")

## 5. NumPy visualization compatibility patch

The repository's visualization code predates NumPy 1.20 and uses the removed `np.float` /
`np.int` aliases and `numpy.core.umath_tests`.

In [ ]:
%cd /content/MotionHiFlow

from pathlib import Path

# A simple "old spelling -> new spelling" replacement table.
PATCHES = {
    "src/visualization/common/quaternion.py": [("np.finfo(np.float).eps", "np.finfo(float).eps")],
    "src/visualization/AnimationStructure.py": [(".astype(np.int)", ".astype(int)")],
    "src/visualization/remove_fs.py": [(".astype(np.float)", ".astype(float)")],
}

for file_name, replacements in PATCHES.items():
    path = Path(file_name)
    if not path.exists():
        print("skipped (missing):", file_name)
        continue
    text = path.read_text()
    for old, new in replacements:
        text = text.replace(old, new)
    path.write_text(text)
    print("patched:", file_name)

# Animation.py needs special handling: umath_tests was removed from NumPy, and np.matmul
# is the equivalent of its matrix_multiply.
animation = Path("src/visualization/Animation.py")
if animation.exists():
    text = animation.read_text()
    text = text.replace("import numpy.core.umath_tests as ut\n", "")
    text = text.replace("ut.matrix_multiply(", "np.matmul(")
    animation.write_text(text)
    print("patched:", animation)

print("\nVisualization compatibility patch completed.")

## 6. Prepare pretrained models and evaluator resources

The official `prepare.sh` covers the evaluator, GloVe and the checkpoints. CLIP is fetched
separately because the `prepare.sh clip` mirror frequently fails inside Colab.

In [ ]:
%cd /content/MotionHiFlow

# --- Official HumanML3D evaluator ---
!mkdir -p deps/evaluators/t2m
!rm -f deps/evaluators/t2m/humanml3d_evaluator.zip
!gdown "19C_eiEr0kMGlYVJy_yFL6_Dhk3RvmwhM" -O deps/evaluators/t2m/humanml3d_evaluator.zip
!unzip -q -o deps/evaluators/t2m/humanml3d_evaluator.zip -d deps/evaluators/t2m

# --- GloVe word vectors, required by the evaluator's text encoder ---
!rm -f deps/glove_data.zip
!gdown "1cmXKUT31pqd7_XpJAiWEo1K81TMYHA5n" -O deps/glove_data.zip
!unzip -q -o deps/glove_data.zip -d .

# The repository looks for GloVe under deps/glove, so mirror the extracted glove/ there.
!mkdir -p deps/glove
!cp -a glove/. deps/glove/

# --- MotionHiFlow pretrained checkpoints ---
!bash prepare.sh pretrained

print("Evaluator, GloVe and checkpoints prepared.")

In [ ]:
%cd /content/MotionHiFlow

from huggingface_hub import snapshot_download

# CLIP text encoder, pulled straight from HuggingFace to avoid the flaky prepare.sh mirror.
snapshot_download(
    repo_id="openai/clip-vit-base-patch32",
    local_dir="/content/MotionHiFlow/deps/clip-vit-base-patch32",
)

print("CLIP files prepared.")

In [ ]:
%cd /content/MotionHiFlow

from pathlib import Path

# Resource self-check: any missing item breaks generation or evaluation later on.
REQUIRED_RESOURCES = [
    Path("deps/evaluators/t2m/Comp_v6_KLD005/meta/mean.npy"),
    Path("deps/evaluators/t2m/Comp_v6_KLD005/meta/std.npy"),
    Path("deps/evaluators/t2m/text_mot_match/model/finest.tar"),
    Path("deps/glove/our_vab_data.npy"),
    Path("deps/glove/our_vab_idx.pkl"),
    Path("deps/glove/our_vab_words.pkl"),
    Path("deps/clip-vit-base-patch32/config.json"),
    Path("logs/t2m_tmdit_16d/checkpoints/net_best_fid.tar"),
    Path("logs/t2m_vae_agcn_16d/checkpoints/net_best_fid.tar"),
]

missing = [str(p) for p in REQUIRED_RESOURCES if not p.exists()]
for p in REQUIRED_RESOURCES:
    print(("[OK]  " if p.exists() else "[MISS]"), p)

if missing:
    raise RuntimeError(f"Resources incomplete — re-run Step 6. Missing: {missing}")

print("\nAll required resources are ready.")

## 7. Load the shared benchmark files

Download the following two files from the team GitHub repository and upload them when
prompted:

- `models/motionhiflow/motionhiflow_utils.py`
- `benchmark/benchmark_definition_v1.0.json`

The files are copied to `/content`, outside the cloned upstream repository.


In [ ]:
%cd /content/MotionHiFlow

import sys
from pathlib import Path
from google.colab import files

MODULE_PATH = Path("/content/motionhiflow_utils.py")
BENCHMARK_PATH = Path("/content/benchmark_definition_v1.0.json")

required_files = {
    MODULE_PATH.name: MODULE_PATH,
    BENCHMARK_PATH.name: BENCHMARK_PATH,
}

missing_names = [
    name for name, path in required_files.items()
    if not path.exists()
]

if missing_names:
    print("Please upload:", missing_names)
    uploaded = files.upload()

    for uploaded_name in uploaded:
        basename = Path(uploaded_name).name
        if basename not in required_files:
            continue

        target = required_files[basename]
        if not target.exists():
            Path(uploaded_name).replace(target)

still_missing = [
    str(path) for path in required_files.values()
    if not path.exists()
]
if still_missing:
    raise FileNotFoundError(
        "Missing required files: " + ", ".join(still_missing)
    )

if str(MODULE_PATH.parent) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH.parent))

%load_ext autoreload
%autoreload 2

import motionhiflow_utils

print("Loaded utility module:", motionhiflow_utils.__file__)
print("Loaded benchmark definition:", BENCHMARK_PATH)


In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Video, display

from motionhiflow_utils import (
    MatchingScorer,
    RunConfig,
    check_left_right_convention,
    check_standardisation,
    render_standard_gif,
    write_prompt_file,
)

# ------------------------------------------------------------------
# Frozen run settings. Change SEED for another approved repetition.
# Keep BENCHMARK_SPLIT="Pilot" until the integration pipeline passes.
# ------------------------------------------------------------------
MODEL_NAME = "MotionHiFlow"
BENCHMARK_FILE_VERSION = "v1.0"
BENCHMARK_SPLIT = "Pilot"       # later change to "Main"
SEED = 1
RUN_VERSION = "v1"
FPS = 20
JOINT_COUNT = 22

split_slug = BENCHMARK_SPLIT.lower()
version_slug = BENCHMARK_FILE_VERSION.replace(".", "_")
RUN_NAME = f"benchmark_{version_slug}_{split_slug}"
EXPORT_NAME = (
    f"motionhiflow_{version_slug}_{split_slug}_seed{SEED}"
)

print("Model             :", MODEL_NAME)
print("Benchmark version :", BENCHMARK_FILE_VERSION)
print("Split             :", BENCHMARK_SPLIT)
print("Seed              :", SEED)
print("Export name       :", EXPORT_NAME)


## 8. Quick qualitative generation (optional)

Run this once after preparing a fresh environment. It confirms that the upstream
generation path works before the official batch starts. This output is not part of the
benchmark result.


In [ ]:
%cd /content/MotionHiFlow

QUICK_PROMPT = "A person walks forward."
QUICK_LENGTH = 100

subprocess.run(
    [
        "bash", "run.sh", "gen", "tmdit",
        "gpu_id=0",
        f"seed={SEED}",
        f"text_prompt={QUICK_PROMPT}",
        f"motion_length={QUICK_LENGTH}",
        "repeat_times=1",
    ],
    cwd="/content/MotionHiFlow",
    check=True,
)


In [ ]:
import glob
import os

videos = sorted(
    glob.glob("/content/MotionHiFlow/outputs/**/*.mp4", recursive=True),
    key=os.path.getmtime,
    reverse=True,
)

if videos:
    print("Latest video:", videos[0])
    display(Video(videos[0], embed=True))
else:
    print("No MP4 found. Check the generation log above.")


## 9. Load the official Pilot manifest

The model receives only the official prompt ID, text and target frame count. Capability,
difficulty and pair metadata are retained for reporting and later evaluation. Paired
prompts remain separate generation requests and are compared after generation using their
shared `pair_id`.


In [ ]:
benchmark = json.loads(
    BENCHMARK_PATH.read_text(encoding="utf-8")
)

selected_prompts = [
    prompt for prompt in benchmark["prompts"]
    if prompt["set"] == BENCHMARK_SPLIT
]

manifest = pd.DataFrame({
    "Benchmark_Order": range(len(selected_prompts)),
    "Prompt_ID": [p["prompt_id"] for p in selected_prompts],
    "Set": [p["set"] for p in selected_prompts],
    "Capability": [p["capability"] for p in selected_prompts],
    "Subtype": [p["subtype"] for p in selected_prompts],
    "Difficulty": [p["difficulty"] for p in selected_prompts],
    "Pair_ID": [p.get("pair_id") for p in selected_prompts],
    "Prompt": [p["text"] for p in selected_prompts],
    "Motion_Length": [
        int(p["target_frames_20fps"])
        for p in selected_prompts
    ],
    "Requirements_JSON": [
        json.dumps(p["requirements"], ensure_ascii=False)
        for p in selected_prompts
    ],
    "Diagnostic_Axes_JSON": [
        json.dumps(p.get("diagnostic_axes", []), ensure_ascii=False)
        for p in selected_prompts
    ],
    "Primary_Scoring_Path_JSON": [
        json.dumps(p.get("primary_scoring_path", []), ensure_ascii=False)
        for p in selected_prompts
    ],
})

if manifest.empty:
    raise RuntimeError(
        f"No prompts found for split: {BENCHMARK_SPLIT}"
    )
if not manifest["Prompt_ID"].is_unique:
    duplicates = manifest.loc[
        manifest["Prompt_ID"].duplicated(), "Prompt_ID"
    ].tolist()
    raise RuntimeError(f"Duplicate Prompt IDs: {duplicates}")

length_counts = (
    manifest["Motion_Length"]
    .value_counts()
    .sort_index()
    .to_dict()
)

if BENCHMARK_SPLIT == "Pilot":
    expected_pilot_lengths = {100: 6, 140: 7, 180: 5}
    if len(manifest) != 18 or length_counts != expected_pilot_lengths:
        raise RuntimeError(
            "The uploaded benchmark JSON is not the approved 18-prompt "
            "Pilot definition. Expected 18 prompts with length counts "
            f"{expected_pilot_lengths}, but found {len(manifest)} prompts "
            f"with {length_counts}. Download the latest "
            "benchmark_definition_v1.0.json from the team repository."
        )

print("Benchmark name   :", benchmark.get("benchmark_name"))
print("Schema version   :", benchmark.get("schema_version"))
print("Definition status:", benchmark.get("status"))
print("Selected prompts :", len(manifest))
print("Length counts    :", length_counts)
print("Difficulty counts:")
print(manifest["Difficulty"].value_counts())

display(manifest)


## 10. Generate the benchmark in length-specific batches

`RunConfig` intentionally represents one motion length. The manifest is therefore split
into 100-, 140- and 180-frame batches. Every batch resets its local row index so that it
matches MotionHiFlow's `sample0`, `sample1`, ... output naming.


In [ ]:
%cd /content/MotionHiFlow

run_contexts = {}
file_manifests = []

for target_frames, batch_manifest in manifest.groupby(
    "Motion_Length", sort=True
):
    target_frames = int(target_frames)
    batch_manifest = batch_manifest.reset_index(drop=True).copy()

    cfg = RunConfig(
        motion_length=target_frames,
        run_name=RUN_NAME,
        run_version=f"{RUN_VERSION}_seed{SEED}",
    ).make_dirs()

    batch_manifest.to_csv(cfg.manifest_csv, index=False)
    write_prompt_file(batch_manifest, cfg.prompt_txt)

    print("\n" + "=" * 72)
    print(
        f"Generating {len(batch_manifest)} prompts "
        f"at {target_frames} frames"
    )
    print(cfg.summary())

    command = [
        "bash", "run.sh", "gen", "tmdit",
        "gpu_id=0",
        f"seed={SEED}",
        f"text_path={cfg.prompt_txt}",
        "repeat_times=1",
        "batch_size=16",
        f"output_dir={cfg.run_output_root}",
    ]

    subprocess.run(
        command,
        cwd=str(cfg.root),
        check=True,
    )

    batch_files = cfg.build_file_manifest(batch_manifest)

    metadata_columns = [
        "Prompt_ID",
        "Benchmark_Order",
        "Set",
        "Capability",
        "Subtype",
        "Difficulty",
        "Pair_ID",
    ]
    batch_files = batch_files.merge(
        batch_manifest[metadata_columns],
        on="Prompt_ID",
        how="left",
    )
    batch_files["Seed"] = SEED
    batch_files["Run_Tag"] = cfg.run_tag

    run_contexts[target_frames] = {
        "cfg": cfg,
        "manifest": batch_manifest,
        "file_manifest": batch_files,
    }
    file_manifests.append(batch_files)

file_manifest = (
    pd.concat(file_manifests, ignore_index=True)
    .sort_values("Benchmark_Order")
    .reset_index(drop=True)
)

print("\nGeneration status:")
print(file_manifest["Status"].value_counts(dropna=False))

failed = file_manifest[file_manifest["Status"] != "OK"]
if not failed.empty:
    raise RuntimeError(
        "Some MotionHiFlow outputs were not located correctly:\n"
        + failed.to_string(index=False)
    )

if len(file_manifest) != len(manifest):
    raise RuntimeError(
        f"Expected {len(manifest)} generated files, "
        f"found {len(file_manifest)}"
    )

GENERATION_MANIFEST_PATH = Path(
    f"/content/{EXPORT_NAME}_generation_manifest.csv"
)
file_manifest.to_csv(GENERATION_MANIFEST_PATH, index=False)
print("Saved:", GENERATION_MANIFEST_PATH)

display(file_manifest)


## 11. Standardise and validate all motions

`motionhiflow_utils.standardise_motionhiflow` applies:

1. whole-sequence ground alignment to `Y = 0`;
2. initial root alignment to `XZ = (0, 0)`;
3. X mirroring for the benchmark convention `+X = Right`, `-X = Left`.

The final representation is `[T, 22, 3]`, 20 fps, metres, `+Y` up and `+Z` forward.


In [ ]:
standard_reports = []

for target_frames, context in run_contexts.items():
    cfg = context["cfg"]
    batch_manifest = context["manifest"]
    batch_files = context["file_manifest"]

    batch_standard = cfg.standardise(batch_files)

    metadata_columns = [
        "Prompt_ID",
        "Benchmark_Order",
        "Set",
        "Capability",
        "Subtype",
        "Difficulty",
        "Pair_ID",
        "Motion_Length",
    ]
    batch_standard = batch_standard.merge(
        batch_manifest[metadata_columns],
        on="Prompt_ID",
        how="left",
    )
    batch_standard["Seed"] = SEED
    batch_standard["Model_Name"] = MODEL_NAME
    batch_standard["Run_Tag"] = cfg.run_tag

    standard_reports.append(batch_standard)

standard_report = (
    pd.concat(standard_reports, ignore_index=True)
    .sort_values("Benchmark_Order")
    .reset_index(drop=True)
)

if len(standard_report) != len(manifest):
    raise RuntimeError(
        f"Expected {len(manifest)} standard motions, "
        f"found {len(standard_report)}"
    )
if not standard_report["Prompt_ID"].is_unique:
    raise RuntimeError("Standard report contains duplicate Prompt IDs.")

missing_ids = sorted(
    set(manifest["Prompt_ID"])
    - set(standard_report["Prompt_ID"])
)
if missing_ids:
    raise RuntimeError(f"Missing standard motions: {missing_ids}")

frame_mismatch = standard_report[
    standard_report["Frames"] != standard_report["Motion_Length"]
]
if not frame_mismatch.empty:
    raise RuntimeError(
        "Generated lengths do not match the benchmark definition:\n"
        + frame_mismatch[
            ["Prompt_ID", "Frames", "Motion_Length"]
        ].to_string(index=False)
    )

check_standardisation(standard_report)
display(standard_report)


In [ ]:
# Directional probes help verify the X-axis convention. A failed probe may indicate
# either an axis problem or a model failure, so it is reported for manual review rather
# than automatically changing mirror_x.
for prompt_id, expected_direction in [
    ("C2-01", "left"),
    ("C2-02", "right"),
]:
    probe = standard_report[
        standard_report["Prompt_ID"] == prompt_id
    ]
    if probe.empty:
        continue

    row = probe.iloc[0]
    motion = np.load(row["Standard_Joint_File"])
    ok, dx = check_left_right_convention(
        motion,
        expect=expected_direction,
    )
    print(
        f"{prompt_id} | expected={expected_direction:<5} | "
        f"ok={ok} | root dX={dx:+.3f}"
    )

print("\nFacing status:")
print(standard_report["Facing_Status"].value_counts(dropna=False))


## 12. Build the framework hand-off package

This package is the boundary between the model-specific generation environment and the
common benchmark framework. Evaluators should consume the standardised joints and use the
official `Prompt_ID` to retrieve requirements from the benchmark JSON.


In [ ]:
EXPORT_ROOT = Path(f"/content/{EXPORT_NAME}")
EXPORT_JOINTS = EXPORT_ROOT / "joints"
EXPORT_REPORTS = EXPORT_ROOT / "reports"
EXPORT_GIFS = EXPORT_ROOT / "gifs"

for directory in [EXPORT_JOINTS, EXPORT_REPORTS, EXPORT_GIFS]:
    directory.mkdir(parents=True, exist_ok=True)

for _, row in standard_report.iterrows():
    source = Path(row["Standard_Joint_File"])
    target = EXPORT_JOINTS / f"{row['Prompt_ID']}.npy"
    shutil.copy2(source, target)

STANDARD_REPORT_PATH = EXPORT_REPORTS / "standardisation_report.csv"
standard_report.to_csv(STANDARD_REPORT_PATH, index=False)

benchmark_sha256 = hashlib.sha256(
    BENCHMARK_PATH.read_bytes()
).hexdigest()

try:
    model_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd="/content/MotionHiFlow",
        text=True,
    ).strip()
except Exception:
    model_revision = None

run_manifest = {
    "model_name": MODEL_NAME,
    "model_revision": model_revision,
    "benchmark_file_version": BENCHMARK_FILE_VERSION,
    "benchmark_name": benchmark.get("benchmark_name"),
    "benchmark_schema_version": benchmark.get("schema_version"),
    "benchmark_status": benchmark.get("status"),
    "benchmark_sha256": benchmark_sha256,
    "split": BENCHMARK_SPLIT,
    "seed": SEED,
    "prompt_count": int(len(standard_report)),
    "length_counts": {
        str(k): int(v) for k, v in length_counts.items()
    },
    "fps": FPS,
    "joint_count": JOINT_COUNT,
    "units": "metres",
    "coordinate_system": {
        "x": "+X=Right, -X=Left",
        "y": "+Y=Up",
        "z": "+Z=Forward, -Z=Backward",
        "initial_root_xz": [0.0, 0.0],
        "ground_y": 0.0,
    },
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "numpy_version": np.__version__,
    "run_tags": {
        str(frames): context["cfg"].run_tag
        for frames, context in run_contexts.items()
    },
}

RUN_MANIFEST_PATH = EXPORT_REPORTS / "run_manifest.json"
RUN_MANIFEST_PATH.write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)

handoff_records = []
for _, row in standard_report.iterrows():
    handoff_records.append({
        "prompt_id": row["Prompt_ID"],
        "model_name": MODEL_NAME,
        "motion_file": f"joints/{row['Prompt_ID']}.npy",
        "source_format": "benchmark_global_xyz",
        "fps": FPS,
        "joint_count": JOINT_COUNT,
        "frames": int(row["Frames"]),
        "target_frames": int(row["Motion_Length"]),
        "seed": SEED,
        "capability": row["Capability"],
        "difficulty": row["Difficulty"],
        "pair_id": (
            None if pd.isna(row["Pair_ID"])
            else row["Pair_ID"]
        ),
        "coordinate_system": "benchmark_global_xyz",
        "up_axis": "+Y",
        "forward_axis": "+Z",
        "x_axis": "+X=Right, -X=Left",
        "units": "metres",
    })

HANDOFF_PATH = EXPORT_REPORTS / "standard_motion_manifest.jsonl"
with HANDOFF_PATH.open("w", encoding="utf-8") as handle:
    for record in handoff_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Export root:", EXPORT_ROOT)
print("Standard joints:", len(list(EXPORT_JOINTS.glob("*.npy"))))
print("Report:", STANDARD_REPORT_PATH)
print("Run manifest:", RUN_MANIFEST_PATH)
print("Framework hand-off:", HANDOFF_PATH)


## 13. Render standardised GIFs (optional)

Rendering is qualitative validation only. It does not change the standard motions and
does not contribute directly to automatic evaluator scores.


In [ ]:
preview_row = standard_report.iloc[0]
preview_path = Path(
    f"/content/{preview_row['Prompt_ID']}_preview.gif"
)

render_standard_gif(
    np.load(preview_row["Standard_Joint_File"]),
    preview_path,
    preview_row["Prompt_ID"],
    preview_row["Prompt"],
    fps=FPS,
)

display(Image(filename=str(preview_path)))


In [ ]:
gif_reports = []

for target_frames, context in run_contexts.items():
    cfg = context["cfg"]
    subset = standard_report[
        standard_report["Motion_Length"] == target_frames
    ].copy()

    batch_gifs = cfg.render(subset, fps=FPS)
    batch_gifs["Motion_Length"] = target_frames
    gif_reports.append(batch_gifs)

gif_report = pd.concat(gif_reports, ignore_index=True)

for _, row in gif_report.iterrows():
    shutil.copy2(
        row["GIF_File"],
        EXPORT_GIFS / f"{row['Prompt_ID']}.gif",
    )

final_report = standard_report.merge(
    gif_report,
    on=["Prompt_ID", "Motion_Length"],
    how="left",
)
final_report.to_csv(STANDARD_REPORT_PATH, index=False)

print("Rendered GIFs:", len(gif_report))
display(gif_report)


## 14. Matching Score (optional diagnostic)

Matching Score is a text-motion embedding distance; lower is better. It is not an atomic
requirement evaluator. Scores are only compared within the same motion length, so the
summary remains stratified by 100, 140 and 180 frames.


In [ ]:
%cd /content/MotionHiFlow

# spaCy provides the POS tags required by HumanML3D text preprocessing.
!python -m spacy download en_core_web_sm -q

scorer = MatchingScorer(
    repo_root="/content/MotionHiFlow",
    feature_dim=263,
)
print("Evaluator ready on", scorer.device)


In [ ]:
score_tables = []

for target_frames, context in run_contexts.items():
    cfg = context["cfg"]
    batch_manifest = context["manifest"]

    batch_scores = cfg.score(
        scorer,
        batch_manifest,
    )
    batch_scores["Motion_Length"] = target_frames
    batch_scores["Seed"] = SEED
    score_tables.append(batch_scores)

matching_scores = pd.concat(score_tables, ignore_index=True)
MATCHING_SCORE_PATH = EXPORT_REPORTS / "matching_scores.csv"
matching_scores.to_csv(MATCHING_SCORE_PATH, index=False)

print("\nLength-stratified Matching Score summary (lower is better):")
display(
    matching_scores.groupby(
        ["Motion_Length", "Capability"]
    )["Matching_Score"]
    .agg(["mean", "count"])
    .round(4)
)

# Add scores to the unified report without requiring GIF rendering.
report_for_merge = (
    final_report.copy()
    if "final_report" in globals()
    else standard_report.copy()
)
report_with_scores = report_for_merge.merge(
    matching_scores[
        [
            "Prompt_ID",
            "Motion_Length",
            "Scored_Frames",
            "Matching_Score",
        ]
    ],
    on=["Prompt_ID", "Motion_Length"],
    how="left",
)
report_with_scores.to_csv(STANDARD_REPORT_PATH, index=False)

print("Saved:", MATCHING_SCORE_PATH)
display(matching_scores)


## 15. Package and download the model hand-off

The ZIP is intended for the common evaluation environment or shared project storage. Do
not commit the ZIP, `.npy` motions or GIF collection to GitHub. Commit only the notebook,
utility module and selected small summary CSV/JSON files.


In [ ]:
ARCHIVE_BASE = Path(f"/content/{EXPORT_NAME}_handoff")
archive_path = shutil.make_archive(
    str(ARCHIVE_BASE),
    "zip",
    root_dir=str(EXPORT_ROOT),
)

print("Created:", archive_path)
print("Archive contents:")
for path in sorted(EXPORT_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(EXPORT_ROOT))

files.download(archive_path)


## 16. Completion checklist

A valid Pilot run should satisfy all of the following:

- 18 unique official Prompt IDs;
- six 100-frame, seven 140-frame and five 180-frame motions;
- one generated and one standardised motion per Prompt ID;
- every motion has shape `[T, 22, 3]` with finite values;
- ground `Y = 0` and initial root `XZ = (0, 0)`;
- coordinate metadata states `+X=Right`, `+Y=Up`, `+Z=Forward`;
- the hand-off ZIP contains `joints/` and `reports/standard_motion_manifest.jsonl`;
- Matching Scores, when run, are reported separately by motion length.

After the Pilot pipeline and common evaluators have been reviewed, change
`BENCHMARK_SPLIT` to `"Main"` and rerun with the same frozen code, checkpoint and approved
seed settings.

### GitHub branch hand-off

Upload this notebook back to `models/motionhiflow/` on the `motionhiflow` branch with a
commit such as:

```text
Update MotionHiFlow pipeline for official Pilot benchmark
```

Do not modify the shared benchmark JSON or framework files from the model branch.
